In [4]:
## Importando libs necessarias para o pipeline
import pandas as pd
import boto3
import io
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import logging
import sys


In [7]:
# 1. Configuração global do sistema de logs
logging.basicConfig(
    level=logging.INFO,  # Registra tudo de INFO para cima (INFO, WARNING, ERROR, CRITICAL)
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        # Cria e salva os logs em um arquivo chamado "app.log"
        logging.FileHandler("app.log", encoding="utf-8"),
        
        # Joga os logs de forma estruturada no terminal
        logging.StreamHandler(sys.stdout)
    ]
)

# 2. Criar o objeto 'logger' que usaremos no código
logger = logging.getLogger("Jornada de Dados")

In [8]:
logger.info("Iniciando o script de extração de dados...")

load_dotenv()

2026-05-27 21:31:54 | INFO     | Jornada de Dados | Iniciando o script de extração de dados...


True

In [2]:
## configurando variaveis de ambiente, todas salvas no arquivos .env na raiz do projeto
S3_ENDPOINT_URL = os.getenv("S3_ENDPOINT_URL")
AWS_REGION = os.getenv("AWS_REGION")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET_NAME = os.getenv("BUCKET_NAME")
DATABASE_URL= os.getenv("DATABASE_URL")
DIRECT_URL= os.getenv("DIRECT_URL")

In [9]:
## Criando conexão com o S3
logger.info(f"Conectando ao S3")
s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)

2026-05-27 21:34:18 | INFO     | Jornada de Dados | Conectando ao S3


In [11]:
logger.info(f"Verificando arquivos no Bucket")
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
arquivos = [obj["Key"] for obj in response["Contents"]]

logger.info(f"{arquivos}")

2026-05-27 21:35:46 | INFO     | Jornada de Dados | Verificando arquivos no Bucket
2026-05-27 21:35:47 | INFO     | Jornada de Dados | ['clientes.parquet', 'preco_competidores.parquet', 'produtos.parquet', 'vendas.parquet']


In [12]:
logger.info(f"Tentando conectar ao banco de dados")
engine = create_engine(DATABASE_URL)

2026-05-27 21:37:10 | INFO     | Jornada de Dados | Tentando conectar ao banco de dados


In [13]:
## Nome das tabelas finais, usando como padrão o nome do arquivo original.
tabelas = ['clientes', 'preco_competidores', 'produtos', 'vendas']
dataframe = {}

In [14]:
## Lendo arquivos parquet
for tabela in tabelas:
    file_key = f"{tabela}.parquet"
    logger.info(f"Lendo arquivo: {file_key}")
    response = s3.get_object(Bucket=BUCKET_NAME, Key=file_key)
    parquet_bytes = response["Body"].read()
    dataframe[tabela] = pd.read_parquet(io.BytesIO(parquet_bytes))

2026-05-27 21:40:07 | INFO     | Jornada de Dados | Lendo arquivo: clientes.parquet
2026-05-27 21:40:09 | INFO     | Jornada de Dados | Lendo arquivo: preco_competidores.parquet
2026-05-27 21:40:10 | INFO     | Jornada de Dados | Lendo arquivo: produtos.parquet
2026-05-27 21:40:10 | INFO     | Jornada de Dados | Lendo arquivo: vendas.parquet


In [15]:
for tabela, df in dataframe.items():
    logger.info(f"Criando tabela: {tabela}")
    df.to_sql(
        tabela,                # Nome da tabela no banco
        engine,                # Engine de conexão
        if_exists="replace",   # Substituir se existir
        index=False,           # Não salvar índice do pandas
    )
    logger.info(f"Tabela {tabela} criada com sucesso!!")


2026-05-27 21:41:48 | INFO     | Jornada de Dados | Criando tabela: clientes
2026-05-27 21:41:59 | INFO     | Jornada de Dados | Tabela clientes criada com sucesso!!
2026-05-27 21:41:59 | INFO     | Jornada de Dados | Criando tabela: preco_competidores
2026-05-27 21:42:06 | INFO     | Jornada de Dados | Tabela preco_competidores criada com sucesso!!
2026-05-27 21:42:06 | INFO     | Jornada de Dados | Criando tabela: produtos
2026-05-27 21:42:12 | INFO     | Jornada de Dados | Tabela produtos criada com sucesso!!
2026-05-27 21:42:12 | INFO     | Jornada de Dados | Criando tabela: vendas
2026-05-27 21:42:23 | INFO     | Jornada de Dados | Tabela vendas criada com sucesso!!


In [16]:
logger.info("\n📊 Verificação final:")
for tabela in tabelas:
    df_verificacao = pd.read_sql_query(f"SELECT COUNT(*) as total FROM {tabela}", engine)
    total = df_verificacao["total"].iloc[0]
    logger.info(f"  ✅ {tabela}: {total} linhas no banco")

# Fechar conexão
engine.dispose()

2026-05-27 21:43:07 | INFO     | Jornada de Dados | 
📊 Verificação final:
2026-05-27 21:43:08 | INFO     | Jornada de Dados |   ✅ clientes: 50 linhas no banco
2026-05-27 21:43:09 | INFO     | Jornada de Dados |   ✅ preco_competidores: 728 linhas no banco
2026-05-27 21:43:10 | INFO     | Jornada de Dados |   ✅ produtos: 215 linhas no banco
2026-05-27 21:43:10 | INFO     | Jornada de Dados |   ✅ vendas: 3020 linhas no banco
